In [10]:
from trot.trial.ghf import make_ghf_trial_ops
from trot.trial.ghf import GhfTrial
from trot.ham.hubbard import HamHubbard
from trot.driver import run_qmc
from trot.core.system import System
from trot.prop.types import QmcParams
from trot.meas.ghf import make_ghf_meas_ops_hubbard
from trot.prop.cpmc_slow import make_prop_ops
from trot.prop import blocks
import jax.numpy as jnp
import numpy as np
from pyscf import gto
from pyscf import ao2mo, scf

L = 8
U = 12
t = 1.0
n = L 
mol = gto.M(verbose=4)
mol.incore_anyway = True
mol.nelec = (L//2,L//2)
inital_state = np.zeros((2,L,L))
for i in range(L//2):
    inital_state[0,2*i,2*i] = 1
    inital_state[1,2*i+1,2*i+1] = 1
print(inital_state)
h1 = np.zeros((n,n))
for i in range(n-1):
    h1[i,i+1] = h1[i+1,i] = -t
eri = np.zeros((n,n,n,n))
for i in range(n):
    eri[i,i,i,i] = U
mf = scf.UHF(mol)
mf.get_hcore = lambda *args: h1
mf.get_ovlp = lambda *args: np.eye(L)
mf._eri = ao2mo.restore(8, eri, L)
mf.kernel(inital_state)


System: uname_result(system='Darwin', node='WC292CX16G', release='25.5.0', version='Darwin Kernel Version 25.5.0: Mon Apr 27 20:38:56 PDT 2026; root:xnu-12377.121.6~2/RELEASE_ARM64_T6000', machine='arm64')  Threads 1
Python 3.12.14 (main, Aug 12 2026, 13:57:54) [Clang 21.0.0 (clang-2100.1.1.101)]
numpy 2.5.3  scipy 1.18.1  h5py 3.16.0
Date: Fri Oct  2 09:52:44 2026
PySCF version 2.14.0
PySCF path  /Users/fnappi/.trot/lib/python3.12/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 0
[INPUT] num. electrons = 0
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom

nuclear repulsion = 0
number of shells = 0
number of NR pGTOs = 0
number of NR cGTOs = 0
basis = sto-3g
ecp = {}
CPU time:       168.25
[[[1. 0. 0. 0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0. 0. 

np.float64(-1.1597070476816338)

In [11]:
sys = System(L,(L//2,L//2),'unrestricted')
params = QmcParams(n_walkers=20,n_blocks=20,n_eql_blocks=10,seed=2)
ham = HamHubbard(jnp.array(h1),U)
orbitals = mf.mo_coeff
nocc = L//2
trial_data = np.block([
    [orbitals[0][:, :L//2],        np.zeros((L, L//2))],
    [np.zeros((L, L//2)),          orbitals[1][:, :L//2]],
])  
trial = GhfTrial(jnp.array(trial_data))
meas_ops = make_ghf_meas_ops_hubbard(sys)
trial_ops = make_ghf_trial_ops(sys)
prop_ops = make_prop_ops(ham,'unrestricted')
prop_ctx = prop_ops.build_prop_ctx(ham, None, params)

run = run_qmc(
    sys=sys,
    params=params,
    ham_data=ham,
    trial_data=trial,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
    prop_ctx=prop_ctx,
    block_fn=blocks.block,
)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/10]   -1.1597070477  2.000000e+01           0       0.0
[eql    2/10]   -1.1678099543  2.000636e+01           0       0.5
[eql    4/10]   -1.1825911848  2.002902e+01           0       1.0
[eql    6/10]   -1.1734259580  2.001759e+01           0       1.0
[eql    8/10]   -1.2071751923  2.009373e+01           0       1.1
[eql   10/10]   -1.1878209544  2.000864e+01           0       1.1

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    2/20]   -1.1721837907  unavailable     -1.1721837907  2.001781e+01           0      0.010       1.1
[blk    4/20]   -1.8803811393   5.009e-01     -2.5449747470  2.133117e+01           0      0.010       1.1
[blk    6/20]   -3.6741436158   1.697e+00     -6.8674955121  2.322646e+01   

In [12]:
result_trot = run.mean_energy
error_trot = run.stderr_energy
print(f"Final energy from trot computation: {result_trot:.5g} ± {error_trot:.3g}")


Final energy from trot computation: -2.2218 ± 0.712


In [13]:
from trot.gmps.driver import make_mps_cpmc_ops
from trot.gmps.utils import sd_to_gmps
from trot.trial.mps import mps_trial_from_sd,make_mps_trial
from trot.prop.types import QmcParamsMps

Ca, Cb = mf.mo_coeff[0][:, :L//2], mf.mo_coeff[1][:, :L//2]
trial = mps_trial_from_sd(Ca,Cb)
# tensors = sd_to_gmps(Ca, Cb, mode="maximal").tensors
# trial = make_mps_trial(tensors,nelec=(L//2, L//2))
params = QmcParamsMps(n_walkers=20, n_blocks=20, n_eql_blocks=10, seed=2,
                      orbital_plan="maximal", walker_channel_chi=None)
ops = make_mps_cpmc_ops(ham, trial, sys, params)
run = run_qmc(sys=sys, params=params, ham_data=ham, trial_data=trial, trial_ops=ops.trial_ops,
              meas_ops=ops.meas_ops, prop_ops=ops.prop_ops,
              prop_ctx=ops.prop_ops.build_prop_ctx(ham, None, params), block_fn=blocks.block)

[chunks] automatic selection unavailable: the backend did not report a device memory limit; using n_chunks=1.

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/10]   -1.1597070477  2.000000e+01           0       0.0
[eql    2/10]   -1.1678099543  2.000636e+01           0      16.5
[eql    4/10]   -1.1825911848  2.002902e+01           0      19.2
[eql    6/10]   -1.1734259580  2.001759e+01           0      21.8
[eql    8/10]   -1.2071751923  2.009373e+01           0      24.2
[eql   10/10]   -1.1878209544  2.000864e+01           0      26.6

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    2/20]   -1.1721837907  unavailable     -1.1721837907  2.001781e+01           0      1.365      29.3
[blk    4/20]   -1.8803811393   5.009e-01     -2.5449747470  2.133117e+01           0      1.290      31.9
[blk    6/20]   -3.6741436158   1.697e+00     -6.8674955121  2.322646e+01   

In [14]:
result_trot_mps = run.mean_energy
error_trot_mps = run.stderr_energy
print(f"Final energy from trot mps computation: {run.mean_energy:.5g} ± {error_trot:.3g}")
print(f"The two computation agree: {np.allclose(result_trot,result_trot_mps)}")

Final energy from trot mps computation: -2.2218 ± 0.712
The two computation agree: True
